# dashboard 分析导出 v1

目标：验证 `src/analysis` 的 window-regime 交叉分段、模型 fidelity、模型 attribution 能进入统一 dashboard export。skill 只保留发散和抽象思路，实际可回放判断放在 `src/` 和本 notebook 产物。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'SPEC.md').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot locate repo root from notebook cwd')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import write_model_matrix_dashboard_export
from src.model.matrix import run_quantbench_matrix_smoke

CARD = Path('research/dashboard_analysis_export_v1')
MATRIX_CARD = CARD / 'matrix_card'
ATTRIBUTION_CARD = Path('research/model_attribution_analysis_v1')
DASHBOARD_PATH = CARD / 'dashboard_export.json'
CARD.mkdir(parents=True, exist_ok=True)

{'card': str(CARD), 'matrix_card': str(MATRIX_CARD), 'dashboard_export': str(DASHBOARD_PATH)}

{'card': 'research/dashboard_analysis_export_v1',
 'matrix_card': 'research/dashboard_analysis_export_v1/matrix_card',
 'dashboard_export': 'research/dashboard_analysis_export_v1/dashboard_export.json'}

## 生成小型模型矩阵

使用 `ridge_mse` 作为 native 基准、`gcn_mse` 作为 graph proxy 代表，验证同一矩阵产物中能写出 `window_regime_metrics.csv`、`model_capabilities.csv` 和 manifest。

In [2]:
outputs = run_quantbench_matrix_smoke(
    card_dir=MATRIX_CARD,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
)
outputs.summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'smoke_completed',
 'model_count': 2,
 'start': '2026-01-01',
 'end': '2026-06-09',
 'train_end': '2026-04-30',
 'test_start': '2026-05-01',
 'test_end': '2026-06-08',
 'top_model': 'ridge_mse',
 'adapter_status': 'paper_strict_adapter',
 'adapter_statuses': ['paper_strict_adapter'],
 'model_adapters': ['gcn_feature_propagation_mse_v1', 'ridge_mse_v1'],
 'model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 2,
  'fidelity_counts': {'native_library': 1, 'proxy_feature_adapter': 1},
  'proxy_models': ['gcn_mse'],
  'native_models': ['ridge_mse'],
  'promotion_blocker_count': 1},
 'analysis_layers': ['regime_metrics',
  'window_metrics',
  'window_regime_metrics',
  'qlib_risk_metrics',
  'model_effect_deltas'],
 'analysis_summary': {'schema_version': 'analysis.regime_window_effect.v1',
  'baseline_model': 'ridge_mse',
  'model_count': 2,
  'regime_count': 4,
  'window_count': 1,
  'window_regime_count': 4,
  'segment_types': ['all', 'regime', 'window', 'wi

## 写出 dashboard export

Dashboard loader 只读取矩阵 card 的白名单 sidecar，并显式接入 attribution card，不扫描散落实验内部文件。

In [3]:
payload = write_model_matrix_dashboard_export(
    MATRIX_CARD,
    output_path=DASHBOARD_PATH,
    top_n=5,
    attribution_card_dir=ATTRIBUTION_CARD,
)

summary = {
    'schema_version': 'dashboard.analysis_export.v1',
    'status': 'completed',
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(DASHBOARD_PATH),
    'dashboard_schema_version': payload['schema_version'],
    'top_models': [row['model'] for row in payload['top_models']],
    'model_fidelity': payload['model_fidelity'],
    'window_regime_rows': len(payload['window_regime_metrics']),
    'window_regime_segments': sorted({row['segment'] for row in payload['window_regime_metrics']}),
    'attribution_status': payload['attribution']['status'],
    'attribution_models': payload['attribution']['models'],
    'artifact_keys': sorted(payload['artifacts'].keys()),
    'risk_notes': [
        'gcn_mse 当前仍是 proxy_feature_adapter，需要后续图数据和论文级实现补强。',
        'window_regime 分段用于发现同一模型在不同时间窗口和市场状态中的失效点。',
    ],
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'dashboard.analysis_export.v1',
 'status': 'completed',
 'matrix_card': 'research/dashboard_analysis_export_v1/matrix_card',
 'dashboard_export': 'research/dashboard_analysis_export_v1/dashboard_export.json',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'top_models': ['ridge_mse', 'gcn_mse'],
 'model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 2,
  'fidelity_counts': {'native_library': 1, 'proxy_feature_adapter': 1},
  'proxy_models': ['gcn_mse'],
  'native_models': ['ridge_mse'],
  'promotion_blocker_count': 1},
 'window_regime_rows': 8,
 'window_regime_segments': ['test::bull',
  'test::downtrend',
  'test::neutral',
  'test::stress'],
 'attribution_status': 'completed',
 'attribution_models': ['ridge_mse', 'xgboost_mse', 'gcn_mse'],
 'artifact_keys': ['attribution_feature',
  'attribution_model_summaries',
  'attribution_summary',
  'attribution_top_features',
  'model_capabilities',
  'model_comparison',
  'model_effect_delta

In [4]:
preview = {
    'schema_version': payload['schema_version'],
    'model_fidelity': payload['model_fidelity']['fidelity_counts'],
    'window_regime_metrics': payload['window_regime_metrics'][:5],
    'model_effect_deltas': payload['model_effect_deltas'][:5],
    'attribution_top_features': payload['attribution']['top_features_by_model'][:5],
    'artifacts': payload['artifacts'],
}
preview

{'schema_version': 'dashboard.model_matrix.v1',
 'model_fidelity': {'native_library': 1, 'proxy_feature_adapter': 1},
 'window_regime_metrics': [{'window': 'test',
   'regime': 'bull',
   'segment': 'test::bull',
   'model': 'gcn_mse',
   'return_pct': 20.47505531469156,
   'benchmark_return_pct': 82.22807991120966,
   'relative_return_pct': -61.7530245965181,
   'max_drawdown_pct': 2.483159094884746,
   'sharpe': 9.084801213781988,
   'avg_position': 0.6558278503493202,
   'turnover': 1.7982047858514916,
   'stability_score': 104.62270901220252},
  {'window': 'test',
   'regime': 'bull',
   'segment': 'test::bull',
   'model': 'ridge_mse',
   'return_pct': 23.0790050483316,
   'benchmark_return_pct': 82.22807991120966,
   'relative_return_pct': -59.14907486287806,
   'max_drawdown_pct': 2.676198522804263,
   'sharpe': 9.559104571330575,
   'avg_position': 0.6867858631587391,
   'turnover': 1.863232759934674,
   'stability_score': 111.59833708826272},
  {'window': 'test',
   'regime': 

## 结论

- 做了什么：将 window-regime 交叉分段固定到 `src/analysis`，并通过模型矩阵写出 `window_regime_metrics.csv`。
- dashboard 结果：`dashboard_export.json` 顶层包含 `model_fidelity`、`model_capabilities`、`window_regime_metrics`、`model_effect_deltas` 和 `attribution`。
- 架构边界：skill 以后只用于抽象思路和生成候选假设；可回放分析、分段比较和页面展示数据必须通过 `src/analysis`、`src/model` 和 `src/dashboard` 的稳定接口生成。
- 风险：本 notebook 只跑 `ridge_mse` 与 `gcn_mse` 的小矩阵；`gcn_mse` 当前是 graph proxy，不能当成论文级 GCN 实盘能力。
- 下一步：把调度 pipeline 的 dashboard refresh 增加可选 attribution 输入，并在页面层增加 window-regime 筛选控件。